# 03 · LLM zero-shot: piloto de modelos

Paso 4.1 del flujo. Se comparan tres LLMs abiertos (Qwen3-8B, Llama-3.1-8B y Gemma 3 12B) en **zero-shot sobre dev** con el mismo prompt y la misma entrada limpia, para elegir el modelo con el que seguir (ajuste del prompt, few-shot y RAG).

- **No se genera texto:** una sola pasada por noticia y se lee la probabilidad del primer token de la respuesta. `p_fake = P(Falsa) / (P(Falsa) + P(Verdadera))`. Así hay AUC y se pueden aplicar las mismas reglas de decisión que a los baselines (umbral 0,5 y rango al 50 %).
- **Criterio de elección:** AUC en dev (no depende del umbral); desempate por F1 con rango, `p_formato` (si el modelo responde en el formato pedido) y tiempo.
- **Test no se usa en este notebook.**

## Entorno

Pensado para **Kaggle**: *Accelerator* → GPU T4 ×2, *Internet* activado y un secreto `HF_TOKEN` (*Add-ons → Secrets*). Llama 3.1 y Gemma 3 son modelos restringidos: hay que aceptar su licencia en Hugging Face con la misma cuenta del token (Gemma es inmediato; Llama requiere aprobación de Meta, que puede tardar horas). Qwen3 no lo necesita.

La celda clona el repo, instala dependencias y genera `data/processed` si no existe. En local (sin GPU) el notebook se ejecuta igual pero se salta los modelos.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
REPO, BRANCH = "https://github.com/Rodrigo-Romay/Practica-TAPLN.git", "feature/LLM"

if IN_KAGGLE:
    ROOT = Path("/kaggle/working/Practica-TAPLN")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers>=4.51", "accelerate", "bitsandbytes"], check=True)
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Sin HF_TOKEN: solo funcionarán los modelos no restringidos (Qwen).", e)
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "data" / "processed" / "train.csv").exists():
    raw = ROOT / "data" / "raw"
    subprocess.run([sys.executable, str(ROOT / "src" / "process_data.py"),
                    "--train", str(raw / "train.xlsx"), "--dev", str(raw / "development.xlsx"),
                    "--test", str(raw / "test.xlsx"), "--out", str(ROOT / "data" / "processed")], check=True)

In [ ]:
import numpy as np
import pandas as pd
import torch

import data_utils as du
import preprocessing as pp
import evaluation as ev
import baselines as bl
import llm

splits = du.load_clean_splits(ROOT)
media_re = pp.media_regex_from_splits(splits)   # nombres de medio de train/dev
dev_in = llm.llm_inputs(splits["dev"], media_re)
OUT = ROOT / "results" / "llm"
OUT.mkdir(parents=True, exist_ok=True)

print(f"dev: {len(dev_in)} noticias, {dev_in.label.mean():.1%} Fake")
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "ninguna")

## Prompt

Mismo prompt para los tres modelos (en `src/llm.py`): rol de verificador, aviso de que se han quitado enlaces y medios y de que las cifras aparecen como `NUM`, titular («no disponible» si falta) y cuerpo cortado a 800 palabras. En Qwen3 se desactiva el modo *thinking*; probarlo queda para el ajuste del prompt.

In [ ]:
print(llm.SYSTEM, "\n")
print(llm.USER.format(headline=dev_in.headline[0], body=dev_in.body[0][:300] + " [...]"))

## Candidatos

| Clave | Modelo | Carga en T4 | Licencia |
|---|---|---|---|
| `qwen3-8b` | Qwen/Qwen3-8B | fp16, repartido en 2×T4 | Apache 2.0 |
| `llama31-8b` | meta-llama/Llama-3.1-8B-Instruct | fp16, repartido en 2×T4 | Llama 3.1 Community |
| `gemma3-12b` | google/gemma-3-12b-it | 4 bits (NF4), cálculo en float32 | Gemma Terms |

Gemma va en 4 bits porque en fp16 ocupa ~24 GB de los 32 disponibles. Además calcula en **float32**: en la primera ejecución, con cálculo en fp16, todas sus puntuaciones salieron `NaN` y sus respuestas vacías. Gemma 3 se entrenó en bf16 y sus activaciones se salen del rango de fp16; la T4 no soporta bf16, así que se usa float32 (más lento, mismos pesos).

Cada modelo guarda `results/llm/piloto_dev_<clave>.csv` y, si la sesión se corta, reanuda desde ahí. Con `EJECUTAR` se puede lanzar un solo modelo por sesión.

In [ ]:
MODELOS = {
    "qwen3-8b":   {"name": "Qwen/Qwen3-8B"},
    "llama31-8b": {"name": "meta-llama/Llama-3.1-8B-Instruct"},
    "gemma3-12b": {"name": "google/gemma-3-12b-it", "quant": "4bit", "dtype": torch.float32},  # fp16 da NaN
}
EJECUTAR = list(MODELOS) if torch.cuda.is_available() else []   # p. ej. ["qwen3-8b"]
BORRAR_CACHE = IN_KAGGLE   # borra cada modelo del disco al terminar con él

In [ ]:
for key in EJECUTAR:
    cfg, path = MODELOS[key], OUT / f"piloto_dev_{key}.csv"
    if path.exists() and len(pd.read_csv(path)) == len(dev_in):
        print(f"{key}: ya hecho"); continue
    print(f"== {key} ({cfg['name']})")
    tok, model = llm.load_model(cfg["name"], cfg.get("quant"), cfg.get("dtype"))
    print("Tokens de respuesta:", {k: [tok.decode([i]) for i in v] for k, v in llm.answer_ids(tok).items()})
    # Comprobación a mano del formato con tres noticias
    for i in range(3):
        p = llm.render(tok, llm.build_messages(dev_in.headline[i], dev_in.body[i]))
        print(f"  ejemplo {i} (label={dev_in.label[i]}): {llm.generate_answer(model, tok, p)!r}")
    llm.run(model, tok, dev_in, path)
    del model, tok
    llm.release(cfg["name"], delete_cache=BORRAR_CACHE)

## Resultados en dev

Se compara con el baseline char-SVM entrenado solo con train (mismo protocolo que en `02_baselines`). Columnas extra: `p_formato` medio (≈1 si el modelo responde «Verdadera»/«Falsa»), número de `nan` y segundos por noticia.

In [ ]:
tr, dev = splits["train"], splits["dev"]
rows = []

svm = bl.make_pipe("char", 1.0, "svm").fit(pp.prepare(tr, media_re), tr.label)
s = bl.get_scores(svm, pp.prepare(dev, media_re))
rows.append({**ev.evaluate(dev.label, ev.predict_threshold(s, 0.0), s, "char-svm (baseline)"), "decision": "umbral"})
rows.append({**ev.evaluate(dev.label, ev.predict_by_rank(s, 0.5), s, "char-svm (baseline)"), "decision": "rango 50 %"})

piloto = {}
for key in MODELOS:
    path = OUT / f"piloto_dev_{key}.csv"
    if not path.exists():
        continue
    r = pd.read_csv(path)
    if r.nan.any():
        print(f"{key}: {int(r.nan.sum())} puntuaciones NaN, se excluye (borrar su CSV y repetir)"); continue
    piloto[key] = r
    s, y = r.p_fake.to_numpy(), r.label.to_numpy()
    extra = {"p_formato": r.p_formato.mean(), "nan": int(r.nan.sum()), "s_noticia": r.segundos.mean()}
    rows.append({**ev.evaluate(y, ev.predict_threshold(s, 0.5), s, key), "decision": "umbral 0,5", **extra})
    rows.append({**ev.evaluate(y, ev.predict_by_rank(s, 0.5), s, key), "decision": "rango 50 %", **extra})

cols = ["modelo", "decision", "f1_fake", "precision", "recall", "pct_pred_fake", "auc", "p_formato", "nan", "s_noticia", "n"]
tabla = ev.results_table(rows)
tabla = tabla[[c for c in cols if c in tabla.columns]]
tabla.to_csv(OUT / "piloto_dev_resumen.csv", index=False)
tabla

### Sesgo de cada modelo

Distribución de `p_fake` por clase real. Si las dos clases quedan pegadas a 0 o a 1, el modelo está sesgado hacia una respuesta: el umbral 0,5 falla aunque el AUC sea bueno.

In [ ]:
for key, r in piloto.items():
    print(f"== {key}")
    display(r.groupby("label").p_fake.describe(percentiles=[.1, .5, .9]).round(3))

## Decisión

*(Rellenar tras ejecutar y pasar al diario.)*

- Modelo elegido:
- AUC / F1 (rango) en dev:
- Motivo:
- Problemas encontrados (formato, `nan`, memoria, tiempo):